In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from tabulate import tabulate as tab

import generate_synthetic_data as gsd


# Read Data and Generate Datasets

In [2]:
data, labels = gsd.read_data_txt(IDs = [10])
data_merged, labels_merged, neutrons_merged, gammas_merged = gsd.merge_cases_together(data, labels)

case    total    gammas    neutrons    ratio (g/n)    Amax total    Amax gammas    Amax neutrons
------  -------  --------  ----------  -------------  ------------  -------------  ---------------
case10  112695   58353     54342       1.1            1.6546        1.6546         0.6824
---     ---      ---       ---         ---            ---           ---            ---
Total   112695   58353     54342       1.1---         ---           ---


### config

In [ ]:
## energy cuts
Vsamples_range=(0.08, 4.0) # 0.08, 4.0
Vpeak_range=(0.05, 999) #(0.05, 999)
late_start = 20 # 30
late_end = 200 # 100
afterpulse_frac = 0.10 #0.08
## Templates
amplitudeV = (0.05, 0.35) # no neutrons after 0.35...
Nbins = 6
## Synthetic Samples
sigma_noise = 0.002
min_voltage = 0.05
max_voltage = 0.35 ## cannot go too high because PSD gets distorted
Npulses_train = 40000 
Npulses_val = 160000

# Case = "allCases"
Case = "Case10"
folder = "Synthetic_Datasets"


### Make Samples

In [4]:
# --------------------------------------------------
# Selection
# --------------------------------------------------
data_sel, labels_sel, cutflow = gsd.energy_selection_tight(data_merged, labels_merged, 
                                                  Vsamples_range, 
                                                  Vpeak_range,
                                                  late_start, 
                                                  late_end, 
                                                  afterpulse_frac)
df = gsd.cutflow_table(cutflow)

# --------------------------------------------------
# Templates
# --------------------------------------------------
neutrons_sel = data_sel[labels_sel == 1]
gammas_sel   = data_sel[labels_sel == 0]
bin_edges = np.linspace(amplitudeV[0], amplitudeV[1], Nbins+1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
print('bin_edges', bin_edges)
print('bin_centers', bin_centers)
print('\nneutrons')
bin_centers, templates_n, templates_n_norm = gsd.make_templates(neutrons_sel, bin_edges, align = False)
print('\ngammas')
bin_centers, templates_g, templates_g_norm = gsd.make_templates(gammas_sel, bin_edges, align = False) ## something here -> align not really working; makes PSD look worse...



Step                       N    eff_abs    eff_rel    N_neutron    eff_neutron    N_gamma    eff_gamma
--------------------  ------  ---------  ---------  -----------  -------------  ---------  -----------
Input                 112695     1.0000     1.0000        54342         1.0000      58353       1.0000
Energy selection      109216     0.9691     0.9691        54203         0.9974      55013       0.9428
Amplitude cut          37942     0.3367     0.3474        12217         0.2254      25725       0.4676
Afterpulse rejection   32444     0.2879     0.8551         6725         0.5505      25719       0.9998
bin_edges [0.05 0.1  0.15 0.2  0.25 0.3  0.35]
bin_centers [0.075 0.125 0.175 0.225 0.275 0.325]

neutrons
 sample shape (6725, 296)
 peak amplitude (min, max) -0.0018 0.3595
 average peak amplitude 0.005924665276800965
 counts per bin: [2876 1956  974  598  247   70]

gammas
 sample shape (25719, 296)
 peak amplitude (min, max) -0.0022 0.4556
 average peak amplitude 0.0049519085

In [5]:
# --------------------------------------------------
# Synthetic Samples Training
# --------------------------------------------------
X_train, Y_train = [], []
print('NEUTRONS')
neutrons_synth, _ = gsd.generate_sample(templates = templates_n_norm, 
                                 bin_centers =  bin_centers, 
                                 Npulses = Npulses_train, 
                                 sigma = sigma_noise,
                                 A_min=min_voltage, 
                                 A_max=max_voltage,
                                 Normalize=True) 
neutrons_label = np.ones(neutrons_synth.shape[0])

print('GAMMAS')
gammas_synth, _ = gsd.generate_sample(templates = templates_g_norm, 
                                 bin_centers =  bin_centers, 
                                 Npulses = Npulses_train, 
                                 sigma = sigma_noise,
                                 A_min=min_voltage, 
                                 A_max=max_voltage,
                                 Normalize=True) 
gammas_label = np.zeros(gammas_synth.shape[0])

print(f'PILEUP')
piluep_synth, time_shifts_pileup_train = gsd.generate_pileup_sample(
                                            Npulses = Npulses_train,
                                            neutron_templates_normalized = templates_n_norm, 
                                            gamma_templates_normalized = templates_g_norm, 
                                            bin_centers = bin_centers,
                                            A_min = min_voltage, A_max = max_voltage, 
                                            noise_sigma = sigma_noise,
                                            Normalize=True 
                                            )
piluep_label = np.ones(piluep_synth.shape[0])*2


X_train = np.concatenate((neutrons_synth, gammas_synth, piluep_synth), axis=0)
Y_train = np.concatenate((neutrons_label, gammas_label, piluep_label), axis=0)
print('\nX shape', X_train.shape)
print('sanity check, Y shape', Y_train.shape)
print('time_shifts shape (pile-up only)', time_shifts_pileup_train.shape)

np.savez(
    f"{folder}/synthetic_training_{Case}_120k_noise_{sigma_noise}.npz",
    X=X_train,
    y=Y_train,
    meta=time_shifts_pileup_train   # any third array (labels, dt, class, etc.)
)



NEUTRONS
Clamped fraction: 0.083975
GAMMAS
Clamped fraction: 0.080925
PILEUP

X shape (120000, 296)
sanity check, Y shape (120000,)
time_shifts shape (pile-up only) (40000,)


In [6]:
# --------------------------------------------------
# Synthetic Samples Validation
# --------------------------------------------------
X_val, Y_val = [], []
print('NEUTRONS')
neutrons_synth, _ = gsd.generate_sample(templates = templates_n_norm, 
                                 bin_centers =  bin_centers, 
                                 Npulses = Npulses_val, 
                                 sigma = sigma_noise,
                                 A_min=min_voltage, 
                                 A_max=max_voltage,
                                 Normalize=False) 
neutrons_label = np.ones(neutrons_synth.shape[0])

print('GAMMAS')
gammas_synth, _ = gsd.generate_sample(templates = templates_g_norm, 
                                 bin_centers =  bin_centers, 
                                 Npulses = Npulses_val, 
                                 sigma = sigma_noise,
                                 A_min=min_voltage, 
                                 A_max=max_voltage,
                                 Normalize=False) 
gammas_label = np.zeros(gammas_synth.shape[0])

print(f'PILEUP')
piluep_synth, time_shifts_pileup_val = gsd.generate_pileup_sample(
                                            Npulses = Npulses_val,
                                            neutron_templates_normalized = templates_n_norm, 
                                            gamma_templates_normalized = templates_g_norm, 
                                            bin_centers = bin_centers,
                                            A_min = min_voltage, A_max = max_voltage, 
                                            noise_sigma = sigma_noise,
                                            Normalize=False 
                                            )
piluep_label = np.ones(piluep_synth.shape[0])*2


X_val = np.concatenate((neutrons_synth, gammas_synth, piluep_synth), axis=0)
Y_val = np.concatenate((neutrons_label, gammas_label, piluep_label), axis=0)
print('\nX shape', X_val.shape)
print('sanity check, Y shape', Y_val.shape)
print('time_shifts shape (pile-up only)', time_shifts_pileup_val.shape)

np.savez(
    f"{folder}/synthetic_validation_{Case}_480k_noise_{sigma_noise}.npz",
    X=X_val,
    y=Y_val,
    meta=time_shifts_pileup_val   # any third array (labels, dt, class, etc.)
)



NEUTRONS
Clamped fraction: 0.08263125
GAMMAS
Clamped fraction: 0.08415
PILEUP

X shape (480000, 296)
sanity check, Y shape (480000,)
time_shifts shape (pile-up only) (160000,)


# Read Synthetic Datasets 

In [7]:
data = np.load(f"../synthetic_data/synthetic_training_{Case}_120k_noise_{noise}.npz")

X = data["X"]
Y = data["y"]
dt = data["meta"]
# Sanity Check
print(np.unique(X == X_train))
print(np.unique(Y == Y_train))
print(np.unique(dt == time_shifts_pileup_train))

NameError: name 'noise' is not defined

In [ ]:
data = np.load(f"../synthetic_data/synthetic_test_{Case}_480k_noise_{noise}.npz")

X = data["X"]
Y = data["y"]
dt = data["meta"]
# Sanity Check
print(np.unique(X == X_test))
print(np.unique(Y == Y_test))
print(np.unique(dt == time_shifts_pileup_test))

[ True]
[ True]
[ True]
